In [74]:
import pandas as pd

df = pd.read_excel("PAN Number Validation Dataset.xlsx")
print(df.head(10))

    Pan_Numbers
0    VGLOD3180G
1    PHOXD7232L
2    MGEPH6532A
3    JJCHK4574O
4    XTQIJ2330L
5    HTJYM3835H
6    YQTAP6661X
7    hvofe5635y
8  hyuij7902r  
9    idsmt3429e


In [75]:
xls = pd.ExcelFile("PAN Number Validation Dataset.xlsx")
print(xls.sheet_names)

['Pan_Numbers']


In [76]:
df = pd.read_excel(
    "PAN Number Validation Dataset.xlsx",
    sheet_name="Pan_Numbers"
)

print(df.head(10))
print(df.shape)

    Pan_Numbers
0    VGLOD3180G
1    PHOXD7232L
2    MGEPH6532A
3    JJCHK4574O
4    XTQIJ2330L
5    HTJYM3835H
6    YQTAP6661X
7    hvofe5635y
8  hyuij7902r  
9    idsmt3429e
(10000, 1)


In [77]:
df = pd.read_excel("PAN Number Validation Dataset.xlsx")

print(repr(df.iloc[0,0]))
print(repr(df.iloc[1,0]))
print(repr(df.iloc[2,0]))

'VGLOD3180G'
'PHOXD7232L'
'MGEPH6532A'


In [78]:
print('Total records=', len(df))
total_records= len(df)

Total records= 10000


In [79]:
df["Pan_Numbers"] = df["Pan_Numbers"].astype('string')

In [80]:
print(df["Pan_Numbers"].dtype)

string


In [81]:
df["Pan_Numbers"] = df["Pan_Numbers"].str.strip()

In [82]:
df["Pan_Numbers"] = df["Pan_Numbers"].str.upper()

In [83]:
df.head(10)

,Pan_Numbers
0,VGLOD3180G
1,PHOXD7232L
2,MGEPH6532A
3,JJCHK4574O
4,XTQIJ2330L
5,HTJYM3835H
6,YQTAP6661X
7,HVOFE5635Y
8,HYUIJ7902R
9,IDSMT3429E


In [84]:
print(df[df["Pan_Numbers"]==''])

     Pan_Numbers
5019            
5020            


In [85]:
print(df[df["Pan_Numbers"].isna()])

     Pan_Numbers
5022        <NA>
5027        <NA>
5033        <NA>
5043        <NA>
5057        <NA>
...          ...
9961        <NA>
9972        <NA>
9986        <NA>
9987        <NA>
9997        <NA>

[965 rows x 1 columns]


In [86]:
df= df.replace({"Pan_Numbers":''}, pd.NA)

In [87]:
print(df[df["Pan_Numbers"]==''])

Empty DataFrame
Columns: [Pan_Numbers]
Index: []


In [88]:
print(df[df["Pan_Numbers"].isna()])

     Pan_Numbers
5019        <NA>
5020        <NA>
5022        <NA>
5027        <NA>
5033        <NA>
...          ...
9961        <NA>
9972        <NA>
9986        <NA>
9987        <NA>
9997        <NA>

[967 rows x 1 columns]


In [89]:
df= df.replace({"Pan_Numbers":''}, pd.NA).dropna(subset="Pan_Numbers")

In [90]:
print('Total records=', len(df))

Total records= 9033


In [91]:
print('Unique Values: ', df["Pan_Numbers"].nunique())

Unique Values:  9025


In [92]:
df = df.drop_duplicates(subset="Pan_Numbers", keep='first')
print('Total records=', len(df))

Total records= 9025


Data Validation

In [93]:
def has_adjacent_repitition(pan):
    for i in range(len(pan)-1):
        if pan[i] == pan[i+1]:
            return True
    return False

In [94]:
print(has_adjacent_repitition('AABCD'))
print(has_adjacent_repitition('FGHHH'))
print(has_adjacent_repitition('ABCDX'))
print(has_adjacent_repitition('MNJPQ'))

True
True
False
False


In [95]:
def is_sequencial(pan):
    for i in range(len(pan)-1):
        if ord(pan[i+1]) - ord(pan[i])!=1:
            return False
    return True
print(is_sequencial('ABCDE'))
print(is_sequencial('MNOPQ'))
print(is_sequencial('ABCXY'))

True
True
False


In [96]:
import re

In [97]:
def is_valid_pan(pan):
    if len(pan)!=10:
        return False
    if not re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', pan):
        return False
    if has_adjacent_repitition(pan):
        return False
    if is_sequencial(pan):
        return False
    return True

In [98]:
df["Status"] = df["Pan_Numbers"].apply(lambda x : "Valid" if is_valid_pan(x)else "Invalid")

In [99]:
df.head(10)

,Pan_Numbers,Status
0,VGLOD3180G,Valid
1,PHOXD7232L,Valid
2,MGEPH6532A,Valid
3,JJCHK4574O,Invalid
4,XTQIJ2330L,Invalid
5,HTJYM3835H,Valid
6,YQTAP6661X,Invalid
7,HVOFE5635Y,Valid
8,HYUIJ7902R,Valid
9,IDSMT3429E,Valid


In [101]:
valid_count=(df["Status"]== 'Valid').sum()
invalid_count=(df["Status"]== 'Invalid').sum()
missing_count = total_records-(valid_count+invalid_count)

In [102]:
valid_count

np.int64(3193)

In [103]:
invalid_count

np.int64(5832)

In [105]:
print("Total Records:", total_records)
print("Valid records:", valid_count)
print("Invalid records:", invalid_count)
print("Missing records:", missing_count)

Total Records: 10000
Valid records: 3193
Invalid records: 5832
Missing records: 975


In [116]:
df_summary= pd.DataFrame({ "TOTAL PROCESSED RECORDS":[total_records],
                           "TOTAL VALID COUNT": [valid_count],
                            "TOTAL INVALID COUNT": [invalid_count],
                            "TOTAL MISSING PAN": [missing_count]})

In [117]:
df_summary.head()

,TOTAL PROCESSED RECORDS,TOTAL VALID COUNT,TOTAL INVALID COUNT,TOTAL MISSING PAN
0,10000,3193,5832,975


In [118]:
with pd.ExcelWriter("PAN VALIDATION RESULT.xlsx") as writer:
    df.to_excel(writer, sheet_name="PAN Validations", index=False)
    df_summary.to_excel(writer, sheet_name="SUMMARY", index=False)